# Anotacoes

**Regras:**
- % defeito = Qtd Defeito / Qtd Amostra (corrigido)

**Pontos a entender:**

- Como classificar os desvios de producao com maiores perdas?
    - Digo isso pois nem sempre aquele desvio mais frequente causou mais perdas


# Importando bibliotecas

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Importando dados

In [2]:
df_hh = pd.read_excel('../data/raw/Desvios por máquina - HH.xlsx', header=2)

df_diario = pd.read_excel('../data/raw/Desvios por máquina - Completo.xlsx', header=2)

c:\Users\evosystem03.ti\Documents\Demanda Carteira\Projeto\predicao-carteira-wheaton\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
c:\Users\evosystem03.ti\Documents\Demanda Carteira\Projeto\predicao-carteira-wheaton\.venv\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


# Visualizando dados

In [3]:
df_hh

,Data wht (dia) amostra,Forno,Maquina,Prefixo,OP Vertech,NQI (amostra),Hora a Hora Wht,Localizacao,Desvio sigla,Qtd Amostra (corrigido),Qtd Defeito,% defeito
0,2026-08-10,A,A1,SB -1035-SWN,198618,F,7,AQ,BOL,12,1,0.083333
1,2026-08-10,A,A1,SB -1035-SWN,198618,F,7,AF,BOL,24,3,0.125000
2,2026-08-10,A,A1,SB -1035-SWN,198618,F,8,AQ,BOL,12,1,0.083333
3,2026-08-10,A,A1,SB -1035-SWN,198618,F,8,AQ,DOB,12,1,0.083333
4,2026-08-10,A,A1,SB -1035-SWN,198618,F,8,AF,BOL,34,12,0.352941
...,...,...,...,...,...,...,...,...,...,...,...,...
2094,2026-08-10,1,11,LB -0586-S,198660,B,6,AF,ATR,8,1,0.125000
2095,2026-08-10,1,11,LB -0586-S,198660,B,6,AF,FFU,8,2,0.250000
2096,2026-08-10,1,11,LB -0586-S,198660,B,6,AF,RAB,8,1,0.125000
2097,2026-08-10,1,11,LB -0586-S,198660,B,6,CF,FFU,250,12,0.048000


In [4]:
df_diario

,Data wht (dia) amostra,Forno,Maquina,Prefixo,OP Vertech,NQI (amostra),Localizacao,Desvio sigla,Qtd Amostra (corrigido),Qtd Defeito,% defeito
0,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,BOL,252,42,0.166667
1,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,COS,12,1,0.083333
2,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,DOB,72,6,0.083333
3,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,FSE,12,3,0.250000
4,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,PED,36,6,0.166667
...,...,...,...,...,...,...,...,...,...,...,...
625,2026-08-10,1,11,LB -0586-S,198660,B,AF,TCT,40,7,0.175000
626,2026-08-10,1,11,LB -0586-S,198660,B,AF,TSU,32,4,0.125000
627,2026-08-10,1,11,LB -0586-S,198660,B,AF,TTO,16,3,0.187500
628,2026-08-10,1,11,LB -0586-S,198660,B,CF,FFU,375,18,0.048000


# Tratando dados

In [5]:
# visualizando variaveis e seus tipos
df_diario.info()

<class 'pandas.DataFrame'>
RangeIndex: 630 entries, 0 to 629
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Data wht (dia) amostra   630 non-null    datetime64[us]
 1   Forno                    630 non-null    str           
 2   Maquina                  630 non-null    str           
 3   Prefixo                  630 non-null    str           
 4   OP Vertech               630 non-null    str           
 5   NQI (amostra)            624 non-null    str           
 6   Localizacao              630 non-null    str           
 7   Desvio sigla             630 non-null    str           
 8   Qtd Amostra (corrigido)  630 non-null    int64         
 9   Qtd Defeito              630 non-null    int64         
 10  % defeito                630 non-null    float64       
dtypes: datetime64[us](1), float64(1), int64(2), str(7)
memory usage: 72.1 KB


In [6]:
# OPs que estamos trabalhando nas bases de métricas das máquinas em producoes
OPs = [
    '198594', '198660', '198592', '198176', '198456', '198618',
    '198698', '198567', '198135', '198659', '198639', '198645',
    '198701', '198704', '198707', '198694', '198658', '198663',
    '198613', '198693', '198546'
]

# Padroniza as OPs como texto sem espaços nos dados
df_diario['OP Vertech'] = (
    df_diario['OP Vertech']
    .astype(str)
    .str.strip()
)

# Padroniza os tipos de controle como texto sem espacos nos dados
df_diario['Desvio sigla'] = (
    df_diario['Desvio sigla']
    .astype(str)
    .str.strip()
)

# Filtra somente as OPs da nossa análise
df_diario = df_diario[
    df_diario['OP Vertech'].isin(OPs)
].copy()

display(df_diario.head(10))


,Data wht (dia) amostra,Forno,Maquina,Prefixo,OP Vertech,NQI (amostra),Localizacao,Desvio sigla,Qtd Amostra (corrigido),Qtd Defeito,% defeito
0,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,BOL,252,42,0.166667
1,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,COS,12,1,0.083333
2,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,DOB,72,6,0.083333
3,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,FSE,12,3,0.250000
4,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,PED,36,6,0.166667
5,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,PIP,24,2,0.083333
6,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,SUO,12,1,0.083333
7,2026-08-10,A,A1,SB -1035-SWN,198618,F,AQ,T -,12,1,0.083333
8,2026-08-10,A,A1,SB -1035-SWN,198618,F,AF,BOA,240,17,0.070833
9,2026-08-10,A,A1,SB -1035-SWN,198618,F,AF,BOL,428,62,0.144860


# Analises

## Maiores desvios por producao

In [7]:
# agrupando desvios (Desvio sigla) por OP Vertech e Localizacao, somando a quantidade de desvios (Qtd. Desvios) e a porcentagem de defeito (% defeito)
df_grouped = df_diario.groupby(['OP Vertech', 'Desvio sigla', 'Localizacao']).agg({
    'Qtd Amostra (corrigido)': 'sum',
    '% defeito': 'sum' # Pode ser mean?
}).reset_index()

display(df_grouped)

,OP Vertech,Desvio sigla,Localizacao,Qtd Amostra (corrigido),% defeito
0,198135,A -,AF,10,0.200000
1,198135,ABO,AF,24,0.041667
2,198135,ADE,AF,24,0.041667
3,198135,BC-,AF,24,0.083333
4,198135,BOA,AF,72,0.041667
...,...,...,...,...,...
597,198707,RAB,AF,48,0.104167
598,198707,RCO,AF,24,0.125000
599,198707,RGA,AF,12,0.083333
600,198707,SUO,AF,108,0.148148


In [8]:
# filtrando uma OP e sua localizacao para somar a % defeito
df_filtered = df_grouped[
    (df_grouped['OP Vertech'] == '198135') &
    (df_grouped['Localizacao'] == 'AF')
]

# somando a % defeito para a OP 198135 e Localizacao AF
total_defect_percentage = df_filtered['% defeito'].sum()

print(f"Percentual total de defeitos da OP 198135 e Localizacao AF: {total_defect_percentage.round(2)} %")

Percentual total de defeitos da OP 198135 e Localizacao AF: 2.13 %


In [9]:
# Remove nulos essenciais e ordena tudo apenas uma vez
df_plot = (
    df_grouped
    .dropna(subset=['OP Vertech', 'Localizacao'])
    .sort_values(
        ['OP Vertech', 'Localizacao', '% defeito'],
        ascending=[True, True, False]
    )
)

# Um gráfico para cada OP Vertech
for op_vertech, df_op in df_plot.groupby('OP Vertech', sort=False):

    # Separa os dados por Localizacao
    dados = {
        str(localizacao): df_local
        for localizacao, df_local
        in df_op.groupby('Localizacao', sort=False)
    }

    if not dados:
        continue

    # Primeira localização exibida
    local_inicial = next(iter(dados))
    df_inicial = dados[local_inicial]

    # Cria o gráfico
    fig = go.Figure(
        go.Bar(
            x=df_inicial['Desvio sigla'],
            y=df_inicial['% defeito'],

            # Texto exibido acima das barras
            text=df_inicial['% defeito'],
            texttemplate='%{y:.2%}',
            textposition='outside',

            cliponaxis=False,

            # Hover
            hovertemplate=(
                '<b>Desvio:</b> %{x}<br>'
                '<b>Defeito:</b> %{y:.2%}'
                '<extra></extra>'
            )
        )
    )

    # Dropdown de Localizacao
    botoes = [
        dict(
            label=localizacao,
            method='update',
            args=[
                {
                    'x': [df_local['Desvio sigla']],
                    'y': [df_local['% defeito']],
                    'text': [df_local['% defeito']]
                },
                {
                    'title.text': (
                        f'Percentual de Defeitos - OP Vertech: {op_vertech}'
                        f'<br>Localização: {localizacao}'
                    )
                }
            ]
        )
        for localizacao, df_local in dados.items()
    ]

    # Layout
    fig.update_layout(
        title=dict(
            text=(
                f'Percentual de Defeitos - OP Vertech: {op_vertech}'
                f'<br>Localização: {local_inicial}'
            ),
            x=0.5
        ),

        # Eixo X
        xaxis=dict(
            title='Tipo de Desvio'
        ),

        # Eixo Y formatado como percentual
        yaxis=dict(
            title='Percentual de Defeitos (%)',
            tickformat='.0%'  # 0.10 -> 10%
        ),

        height=600,
        showlegend=False,
        bargap=0.2,

        margin=dict(
            t=130
        ),

        # Dropdown
        updatemenus=[
            dict(
                buttons=botoes,
                direction='down',
                showactive=True,
                x=0,
                y=1.15,
                xanchor='left',
                yanchor='top'
            )
        ],

        # Texto "Localização:"
        annotations=[
            dict(
                text='<b>Localização:</b>',
                x=0,
                y=1.22,
                xref='paper',
                yref='paper',
                showarrow=False,
                xanchor='left'
            )
        ]
    )

    fig.show()

In [12]:
df_grouped_AF = df_grouped[df_grouped['Localizacao'] == 'AF'].copy()

# Remove nulos essenciais e ordena
df_plot_AF = (
    df_grouped_AF
    .dropna(subset=['OP Vertech'])
    .sort_values(
        ['OP Vertech', '% defeito'],
        ascending=[True, False]
    )
)

# Um gráfico para cada OP Vertech
for op_vertech, df_op in df_plot_AF.groupby('OP Vertech', sort=False):

    if df_op.empty:
        continue

    # Maior valor SOMENTE desta OP
    valor_max = df_op['% defeito'].max()

    # Cria o gráfico
    fig = go.Figure(
        go.Bar(
            x=df_op['Desvio sigla'],
            y=df_op['% defeito'],

            # Cor gradual dentro de cada gráfico
            marker=dict(
                color=df_op['% defeito'],
                colorscale='Reds',
                cmin=0,
                cmax=valor_max,
                showscale=False
            ),

            # Texto exibido acima das barras
            text=df_op['% defeito'],
            texttemplate='%{y:.2%}',
            textposition='outside',

            cliponaxis=False,

            # Hover
            hovertemplate=(
                '<b>Desvio:</b> %{x}<br>'
                '<b>Defeito:</b> %{y:.2%}'
                '<extra></extra>'
            )
        )
    )

    # Layout
    fig.update_layout(
        title=dict(
            text=(
                f'Percentual de Defeitos - OP Vertech: {op_vertech}'
                f'<br>Localização: AF'
            ),
            x=0.5
        ),

        xaxis=dict(
            title='Tipo de Desvio'
        ),

        yaxis=dict(
            title='Percentual de Defeitos (%)',
            tickformat='.0%'
        ),

        height=600,
        showlegend=False,
        bargap=0.2,

        margin=dict(
            t=100
        )
    )

    fig.show()

In [16]:
# Remove OPs nulas, ordena do maior para o menor percentual
# dentro de cada OP e mantém somente o TOP 5 de cada OP.
df_plot_AF = (
    df_grouped_AF
    .dropna(subset=['OP Vertech'])
    .sort_values(
        ['OP Vertech', '% defeito'],
        ascending=[True, False]
    )
    .groupby(
        'OP Vertech',
        sort=False,
        observed=True
    )
    .head(5)
)

hover_template = (
    '<b>Desvio:</b> %{x}<br>'
    '<b>Defeito:</b> %{y:.2%}'
    '<extra></extra>'
)

layout_base = dict(
    xaxis=dict(
        title='Tipo de Desvio'
    ),

    yaxis=dict(
        title='Percentual de Defeitos (%)',
        tickformat='.0%'
    ),

    height=600,
    showlegend=False,
    bargap=0.2,

    margin=dict(
        t=100
    )
)

for op_vertech, df_op in df_plot_AF.groupby(
    'OP Vertech',
    sort=False,
    observed=True
):

    # Converte apenas as colunas utilizadas para arrays
    desvios = df_op['Desvio sigla'].to_numpy(copy=False)
    defeitos = df_op['% defeito'].to_numpy(copy=False)

    # Maior percentual desta OP
    valor_max = defeitos.max()

    # Cria o gráfico
    fig = go.Figure(
        go.Bar(
            x=desvios,
            y=defeitos,

            # Cor proporcional ao percentual
            marker=dict(
                color=defeitos,
                colorscale='Reds',
                cmin=0,
                cmax=valor_max,
                showscale=False
            ),

            # Texto acima das barras
            texttemplate='%{y:.2%}',
            textposition='outside',

            cliponaxis=False,

            # Hover
            hovertemplate=hover_template
        )
    )

    # Layout
    fig.update_layout(
        **layout_base,

        title=dict(
            text=(
                f'Top 5 Percentual de Defeitos - OP Vertech: {op_vertech}'
                '<br>Localização: AF'
            ),
            x=0.5
        )
    )

    fig.show()